In [1]:
input_txt <- "_SaD_000.txt"
output_file <- "consolidado_kaggle.csv"

if (!file.exists(input_txt)) {
  cat("❌ Error: No se encontró el archivo '", input_txt, "' en la carpeta.\n")
} else {
  # 1. Leer el archivo línea por línea eliminando espacios vacíos extra
  lineas <- readLines(input_txt, warn = FALSE)
  lineas <- trimws(lineas)
  lineas <- lineas[lineas != ""] # Descartar líneas completamente vacías
  
  # Listas temporales para ir guardando lo que encontremos
  archivos <- c()
  envios_v <- c()
  semillas <- c()
  scores   <- c()
  
  # 2. Recorrer las líneas buscando los bloques
  i <- 1
  while (i <= length(lineas)) {
    # Si la línea actual termina en .csv, encontramos el inicio de un bloque
    if (grepl("\\.csv$", lineas[i])) {
      archivo_actual <- lineas[i]
      
      # La siguiente línea debería tener la info de envios y semilla
      info_linea <- lineas[i + 1]
      
      # La que le sigue debería tener el score numérico
      score_linea <- lineas[i + 2]
      
      # Extraer envios y semillas usando expresiones regulares simples
      envios_match <- regmatches(info_linea, regexec("envios=(\\d+)", info_linea))[[1]]
      semilla_match <- regmatches(info_linea, regexec("semilla=(\\d+)", info_linea))[[1]]
      
      # Verificar si pudimos extraer todo correctamente para este bloque
      if (length(envios_match) >= 2 && length(semilla_match) >= 2 && !is.na(as.numeric(score_linea))) {
        archivos <- c(archivos, archivo_actual)
        envios_v <- c(envios_v, as.integer(envios_match[2]))
        semillas <- c(semillas, as.integer(semilla_match[2]))
        scores   <- c(scores, as.numeric(score_linea))
        
        i <- i + 3 # Avanzamos al siguiente bloque completo
      } else {
        i <- i + 1 # Si falló el formato de este bloque, avanzamos de a una línea
      }
    } else {
      i <- i + 1
    }
  }
  
  # 3. Armar el Dataframe si encontramos datos
  if (length(archivos) > 0) {
    df_kaggle <- data.frame(
      archivo_origen = archivos,
      envios         = envios_v,
      semilla        = semillas,
      public_score   = scores,
      stringsAsFactors = FALSE
    )
    
    # 4. Limpiar el nombre del experimento
    df_kaggle$experimento <- gsub("\\.csv$", "", df_kaggle$archivo_origen)
    df_kaggle$experimento <- gsub("_s\\d+_\\d+$", "", df_kaggle$experimento)
    
    # Reordenar columnas
    df_kaggle <- df_kaggle[, c("experimento", "archivo_origen", "envios", "semilla", "public_score")]
    
    # 5. Guardar a CSV
    write.csv(df_kaggle, output_file, row.names = FALSE, fileEncoding = "UTF-8")
    
    cat("✨ ¡Proceso completado con éxito!\n")
    cat("📂 Se guardaron", nrow(df_kaggle), "registros en '", output_file, "'.\n")
    
    print(head(df_kaggle, 10))
  } else {
    cat("⚠️ No se pudo extraer información. Revisa si el archivo 'datos_kaggle.txt' tiene el texto copiado de forma consecutiva.\n")
  }
}


✨ ¡Proceso completado con éxito!
📂 Se guardaron 369 registros en ' consolidado_kaggle.csv '.
                          experimento
1  KAz635_WorkFlow_01_senior_NHC_B011
2  KAz635_WorkFlow_01_senior_NHC_B011
3  KAz635_WorkFlow_01_senior_NHC_B011
4  KAz635_WorkFlow_01_senior_NHC_B011
5  KAz635_WorkFlow_01_senior_NHC_B011
6  KAz635_WorkFlow_01_senior_NHC_B011
7  KAz635_WorkFlow_01_senior_NHC_B011
8  KAz635_WorkFlow_01_senior_NHC_B011
9  KAz635_WorkFlow_01_senior_NHC_B011
10 KAz635_WorkFlow_01_senior_NHC_B010
                                    archivo_origen envios semilla public_score
1  KAz635_WorkFlow_01_senior_NHC_B011_s5_14000.csv  14000  100069      407.081
2  KAz635_WorkFlow_01_senior_NHC_B011_s5_13500.csv  13500  100069      416.998
3  KAz635_WorkFlow_01_senior_NHC_B011_s5_13000.csv  13000  100069      427.498
4  KAz635_WorkFlow_01_senior_NHC_B011_s5_12500.csv  12500  100069      423.414
5  KAz635_WorkFlow_01_senior_NHC_B011_s5_12000.csv  12000  100069      429.081
6  KAz635_WorkF

In [2]:
library(data.table)

# 1. Leer el CSV consolidado que ejecutamos en el paso anterior
archivo_input <- "consolidado_kaggle.csv"

if (!file.exists(archivo_input)) {
  stop(paste("❌ Error: No se encontró el archivo", archivo_input, "en el directorio actual."))
}

message("Cargando datos desde el CSV consolidado local...")
df_scores <- fread(archivo_input)

# 2. Adaptar y mapear las columnas existentes a la lógica de data.table
# Extraemos el 'semillerio' (ej: el número después de "_s") desde el nombre del archivo de origen
df_scores[, semillerio := as.integer(sub("^.*_s([0-9]{1,2})_.*$", "\\1", archivo_origen))]

# Renombramos public_score para mantener compatibilidad con tu código histórico
setnames(df_scores, "public_score", "publicScore")

# 3. Conservar columnas necesarias y asegurar tipo numérico
df_scores_aux <- df_scores[, .(experimento, semillerio, semilla, envios, publicScore = as.numeric(publicScore))]

# 4. Transponer los envíos a columnas (agregando semillerio al cruce)
df_scores_final <- dcast(
  df_scores_aux, 
  experimento + semillerio + semilla ~ envios, 
  value.var = "publicScore", 
  fun.aggregate = function(x) if(length(x) == 0) NA_real_ else max(x, na.rm = TRUE)
)

# 5. Identificar columnas de envíos mapeadas para realizar los cálculos
columnas_numericas <- setdiff(names(df_scores_final), c("experimento", "semillerio", "semilla"))

# Limpieza de -Inf residuales por seguridad
for (col in columnas_numericas) {
  set(df_scores_final, i = which(df_scores_final[[col]] == -Inf), j = col, value = NA_real_)
}

columnas_ordenadas <- as.character(sort(as.numeric(columnas_numericas)))

# --- Cálculo de métricas por fila ---
# Promedio y Máximo de los envíos transpuestos
df_scores_final[, promedio := rowMeans(.SD, na.rm = TRUE), .SDcols = columnas_ordenadas]
df_scores_final[, maximo := do.call(pmax, c(.SD, na.rm = TRUE)), .SDcols = columnas_ordenadas]

# Diferencia redondeada a 2 decimales
df_scores_final[, dif_prom_max := round(promedio - maximo, 2)]
# -----------------------------------------------------------

# Renombrar columnas de envíos para la presentación final (Viejo = Nuevo)
nuevos_nombres <- paste("envios", columnas_ordenadas)
setnames(df_scores_final, old = columnas_ordenadas, new = nuevos_nombres)

# 6. Orden e indexación final de las columnas del reporte
columnas_finales <- c("experimento", "semillerio", "semilla", nuevos_nombres, "promedio", "maximo", "dif_prom_max")
setcolorder(df_scores_final, columnas_finales)
setorder(df_scores_final, experimento, semillerio, semilla)

# Guardar matriz final en un nuevo CSV para tener el reporte a mano
write.csv(df_scores_final, "reporte_matriz_experimentos.csv", row.names = FALSE, fileEncoding = "UTF-8")
cat("✨ ¡Matriz generada! Reporte guardado en 'reporte_matriz_experimentos.csv'\n")

# Mostrar el resultado final completo en la Notebook
df_scores_final


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Cargando datos desde el CSV consolidado local...



✨ ¡Matriz generada! Reporte guardado en 'reporte_matriz_experimentos.csv'


experimento,semillerio,semilla,envios 8000,envios 8500,envios 9000,envios 9500,envios 10000,envios 10500,envios 11000,envios 11500,envios 12000,envios 12500,envios 13000,envios 13500,envios 14000,promedio,maximo,dif_prom_max
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
KA9500_EX06_NHC_000,10,100019,429.248,427.498,421.414,428.164,438.081,436.081,434.081,436.748,431.248,NA,NA,NA,NA,431.3959,438.081,-6.69
KAz635_WorkFlow_01_senior_NHC001,10,100019,413.748,424.748,429.331,439.914,441.248,431.414,428.664,429.248,431.164,NA,NA,NA,NA,429.9421,441.248,-11.31
KAz635_WorkFlow_01_senior_NHC002,10,100019,427.998,424.331,430.998,438.498,432.998,434.831,440.581,436.831,433.748,NA,NA,NA,NA,433.4238,440.581,-7.16
KAz635_WorkFlow_01_senior_NHC003,10,100019,421.581,419.414,417.998,422.914,432.914,433.664,437.414,427.998,437.331,NA,NA,NA,NA,427.9142,437.414,-9.50
KAz635_WorkFlow_01_senior_NHC004,10,100019,441.331,439.331,437.664,434.914,436.331,436.498,433.664,427.081,428.331,NA,NA,NA,NA,435.0161,441.331,-6.31
KAz635_WorkFlow_01_senior_NHC005,10,100019,434.081,436.081,433.914,427.831,431.664,444.164,444.914,436.164,434.248,NA,NA,NA,NA,435.8957,444.914,-9.02
KAz635_WorkFlow_01_senior_NHC006,10,100019,418.331,416.748,432.164,438.081,444.664,444.831,442.914,450.664,440.331,NA,NA,NA,NA,436.5253,450.664,-14.14
KAz635_WorkFlow_01_senior_NHC007,10,100019,418.331,421.914,437.081,440.581,431.414,429.414,430.664,426.831,429.748,NA,NA,NA,NA,429.5531,440.581,-11.03
KAz635_WorkFlow_01_senior_NHC008,10,100019,431.248,432.414,436.748,439.831,441.081,435.831,435.164,439.164,436.498,NA,NA,NA,NA,436.4421,441.081,-4.64


In [3]:
system2("kaggle", args = c("competitions", "submissions", "labo-1-ba-2026-analista-sr", "-v"), stdout = TRUE, stderr = TRUE)


[1] "ref,fileName,date,description,status,publicScore,privateScore\r"                                                                                      
 [2] "57709516,KAz635_WorkFlow_01_senior_NHC_B011_s5_14000.csv,2026-10-27 17:57:32.397000,envios=14000  semilla=100069,SubmissionStatus.COMPLETE,407.081,\r"
 [3] "57709497,KAz635_WorkFlow_01_senior_NHC_B011_s5_13500.csv,2026-10-27 17:56:59.943000,envios=13500  semilla=100069,SubmissionStatus.COMPLETE,416.998,\r"
 [4] "57709482,KAz635_WorkFlow_01_senior_NHC_B011_s5_13000.csv,2026-10-27 17:56:27.637000,envios=13000  semilla=100069,SubmissionStatus.COMPLETE,427.498,\r"
 [5] "57709466,KAz635_WorkFlow_01_senior_NHC_B011_s5_12500.csv,2026-10-27 17:55:55.257000,envios=12500  semilla=100069,SubmissionStatus.COMPLETE,423.414,\r"
 [6] "57709451,KAz635_WorkFlow_01_senior_NHC_B011_s5_12000.csv,2026-10-27 17:55:22.970000,envios=12000  semilla=100069,SubmissionStatus.COMPLETE,429.081,\r"
 [7] "57709436,KAz635_WorkFlow_01_senior_NHC_B011_s5_11500.csv,2026-10-27 17:54:50.683000,envios=11500  semilla=100069,SubmissionStatus.COMPLETE,434.081,\r"
 [8] "57709422,KAz635_WorkFlow_01_senior_NHC_B011_s5_11000.csv,2026-10-27 17:54:18.130000,envios=11000  semilla=100069,SubmissionStatus.COMPLETE,439.498,\r"
 [9] "57709405,KAz635_WorkFlow_01_senior_NHC_B011_s5_10500.csv,2026-10-27 17:53:45.827000,envios=10500  semilla=100069,SubmissionStatus.COMPLETE,442.831,\r"
[10] "57709387,KAz635_WorkFlow_01_senior_NHC_B011_s5_10000.csv,2026-10-27 17:53:13.590000,envios=10000  semilla=100069,SubmissionStatus.COMPLETE,435.414,\r"
[11] "57704321,KAz635_WorkFlow_01_senior_NHC_B010_s5_14000.csv,2026-10-27 14:40:27.270000,envios=14000  semilla=100057,SubmissionStatus.COMPLETE,408.665,\r"
[12] "57704315,KAz635_WorkFlow_01_senior_NHC_B010_s5_13500.csv,2026-10-27 14:39:55.123000,envios=13500  semilla=100057,SubmissionStatus.COMPLETE,421.748,\r"
[13] "57704308,KAz635_WorkFlow_01_senior_NHC_B010_s5_13000.csv,2026-10-27 14:39:23.060000,envios=13000  semilla=100057,SubmissionStatus.COMPLETE,431.081,\r"
[14] "57704296,KAz635_WorkFlow_01_senior_NHC_B010_s5_12500.csv,2026-10-27 14:38:50.880000,envios=12500  semilla=100057,SubmissionStatus.COMPLETE,430.748,\r"
[15] "57704280,KAz635_WorkFlow_01_senior_NHC_B010_s5_12000.csv,2026-10-27 14:38:18.513000,envios=12000  semilla=100057,SubmissionStatus.COMPLETE,435.998,\r"
[16] "57704261,KAz635_WorkFlow_01_senior_NHC_B010_s5_11500.csv,2026-10-27 14:37:46.457000,envios=11500  semilla=100057,SubmissionStatus.COMPLETE,438.664,\r"
[17] "57704251,KAz635_WorkFlow_01_senior_NHC_B010_s5_11000.csv,2026-10-27 14:37:14.277000,envios=11000  semilla=100057,SubmissionStatus.COMPLETE,436.998,\r"
[18] "57704239,KAz635_WorkFlow_01_senior_NHC_B010_s5_10500.csv,2026-10-27 14:36:42.247000,envios=10500  semilla=100057,SubmissionStatus.COMPLETE,436.831,\r"
[19] "57704215,KAz635_WorkFlow_01_senior_NHC_B010_s5_10000.csv,2026-10-27 14:36:10.133000,envios=10000  semilla=100057,SubmissionStatus.COMPLETE,433.414,\r"
[20] "57701685,KAz635_WorkFlow_01_senior_NHC_B009_s5_14000.csv,2026-10-27 12:54:35.477000,envios=14000  semilla=100049,SubmissionStatus.COMPLETE,418.081,\r"
[21] "57701665,KAz635_WorkFlow_01_senior_NHC_B009_s5_13500.csv,2026-10-27 12:54:03.393000,envios=13500  semilla=100049,SubmissionStatus.COMPLETE,420.081,\r"
[22] "57701653,KAz635_WorkFlow_01_senior_NHC_B009_s5_13000.csv,2026-10-27 12:53:31.310000,envios=13000  semilla=100049,SubmissionStatus.COMPLETE,429.581,\r"
[23] "57701634,KAz635_WorkFlow_01_senior_NHC_B009_s5_12500.csv,2026-10-27 12:52:59.100000,envios=12500  semilla=100049,SubmissionStatus.COMPLETE,434.748,\r"
[24] "57701622,KAz635_WorkFlow_01_senior_NHC_B009_s5_12000.csv,2026-10-27 12:52:27.087000,envios=12000  semilla=100049,SubmissionStatus.COMPLETE,437.164,\r"
[25] "57701610,KAz635_WorkFlow_01_senior_NHC_B009_s5_11500.csv,2026-10-27 12:51:55.040000,envios=11500  semilla=100049,SubmissionStatus.COMPLETE,445.831,\r"
[26] "57701595,KAz635_WorkFlow_01_senior_NHC_B009_s5_11000.csv,2026-10-27 12